In [ ]:
import numpy as np
import MDAnalysis as mda
from graph_utils import live_plot, live_plot_multi
import vdos as vd

In [ ]:
u = mda.Universe("../md/mda.tpr", "imd://localhost:8888",buffersize = 10*1024*1024)
sel = u.select_atoms("resname SOL")
nDelays = 125 # Size of ring buffer (125 x 0.008 ps = 1 ps)
analysis = vd.vdos(sel,nDelays) # initialize analysis object

plot = live_plot_multi(
    title="Center-of-Mass Velocity Auto Correlation Function (VACF)",
    xaxLabel="delay time (ps)", 
    yaxLabel="normalized VACF", 
    dataLabels=["average", "single molecule"])
# fixed axes for visualization
plot["ax"].set_xlim(0, 1)
plot["ax"].set_ylim(-0.3, 1.0)

vd.vdosLib.omp_set_num_threads(3) # parallel C-code (openMP)
tStep = 0
for ts in u.trajectory:
    analysis.single_frame(tStep,ts.time) # process frame
    if (tStep > 0) and (tStep % nDelays == 0): # update plot every 'nDelays' steps
        analysis.copyResidueList()
        analysis.postProcess(analysis.residueListCopy,mode = "total+single")
        time_ps     = np.array(analysis.tau) # time axis (ps)
        # combine translational VACFs components for x,y,z axes
        ## average over all molecules
        vacf_tr     = analysis.trVACF[0][0] + analysis.trVACF[0][1] + analysis.trVACF[0][2]
        ## single residue/molecule
        vacf_tr_mol = analysis.trVACF[1][0] + analysis.trVACF[1][1] + analysis.trVACF[1][2]
        # normalize VACFs
        vacf_tr     = vacf_tr / vacf_tr[0]
        vacf_tr_mol = vacf_tr_mol / vacf_tr_mol[0]
        # update plot
        plot['update'](time_ps, [vacf_tr,vacf_tr_mol])
    tStep += 1

In [ ]:
u = mda.Universe("../md/mda.tpr", "imd://localhost:8888",buffersize = 10*1024*1024)
sel = u.select_atoms("resname SOL")
nDelays = 75 # Size of ring buffer (75 x 0.008 ps = 0.6 ps)
analysis = vd.vdos(sel,nDelays)

plot = live_plot_multi(
    title="Rotational Velocity Auto Correlation Function (VACF)", 
    xaxLabel="delay time (ps)", 
    yaxLabel="normalized VACF", 
    dataLabels=["average", "single molecule"])
# fixed axes for visualization
plot["ax"].set_xlim(0, 0.6)
plot["ax"].set_ylim(-0.7, 1.0)

vd.vdosLib.omp_set_num_threads(2) # parallel C-code (openMP)
tStep = 0
for ts in u.trajectory:
    analysis.single_frame(tStep,ts.time) # process frame
    if (tStep > 0) and (tStep % nDelays == 0): # update plot every 'nDelays' steps
        analysis.copyResidueList()
        analysis.postProcess(analysis.residueListCopy,mode = "total+single")
        time_ps      = np.array(analysis.tau) # time axis(ps)
        # combine rotational VACFs components for 3 axes
        ## average over all molecules
        vacf_rot     = analysis.rotVACF[0][0] + analysis.rotVACF[0][1] + analysis.rotVACF[0][2]
        ## single residue/molecule
        vacf_rot_mol = analysis.rotVACF[1][0] + analysis.rotVACF[1][1] + analysis.rotVACF[1][2]
        # normalize VACFs
        vacf_rot     = vacf_rot / vacf_rot[0]
        vacf_rot_mol = vacf_rot_mol / vacf_rot_mol[0]
        # update plot
        plot['update'](time_ps, [vacf_rot,vacf_rot_mol])
    tStep += 1